# Exercise 7: Normalised coordinates and the tune

**Block 2: Twiss, ellipses and emittance**

**Learning objectives**
* Transform to Courant–Snyder normalised coordinates, where every ellipse becomes a circle.
* Measure the tune in two ways: from the rotation angle per turn, and from a Fourier spectrum (FFT).
* Understand why only the **fractional** part of the tune can be measured at one place (aliasing),
  and why a single position signal cannot tell $Q$ from $1 − Q$.

**Motivation.** Tilted ellipses of different shapes everywhere in the ring are awkward. A change of
coordinates turns all of them into circles, and one turn of the ring becomes a simple rotation by
the angle $\mu = 2\pi Q$. From then on, "phase space" pictures are much easier to read: this is why
all later exercises plot normalised coordinates.

> **Student version.** Fill in the cells marked `# YOUR CODE HERE` and run the check cells after them: they tell you when you are right.

In [ ]:
# --- setup: make `beamlib` importable --------------------------------------
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "beamlib").is_dir())
sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from beamlib import elements as E, optics as O, lattices as L, tracking as T, diagnostics as D, plotting as P
from beamlib.plotting import COLORS, MM
from beamlib.widgets import interactive_plot
P.setup()

ring = L.fodo_ring()
ringB = ring[2:] + ring[:2]                    # observe at location B (alpha != 0), as in Exercise 6
M_B = O.one_turn_matrix(ringB)[:4, :4]
one_turn = E.LinearMap(M_B).track
tw = O.twiss_from_matrix(M_B[:2, :2])
beta, alpha, Q_true = tw["beta"], tw["alpha"], tw["Q"]
print(f"beta = {beta:.3f} m, alpha = {alpha:.3f}, fractional tune from the matrix: {Q_true:.10f}")

## 1. Normalised coordinates

$$\hat{x} = \frac{x}{\sqrt{\beta}}, \qquad \hat{p} = \frac{\alpha\,x + \beta\,p_x}{\sqrt{\beta}} .$$

Then $\varepsilon_i = \hat{x}^2 + \hat{p}^2$: the ellipse becomes a circle of radius $\sqrt{\varepsilon_i}$, and one turn is a clockwise
rotation by $\mu$. (\hat{x}^2 has units of $\sqrt{\text{m}}$; in plots we show $\sqrt{\mu\text{m}}$, so that $\hat{x}^2 + \hat{p}^2$ is directly
in mm mrad or $\mu\text{m}$ in more relaxed notation.)

**Task.** Implement `normalise`.

In [ ]:
def normalise(x, px, beta: float, alpha: float):
    """Courant-Snyder normalised coordinates (x_hat, p_hat) [sqrt(m)]."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
p0 = np.array([[1.5e-3], [0.2e-3], [0.0], [0.0]])
res = T.track(p0, one_turn, 1024)
x, px = res.x[:, 0], res.px[:, 0]
xh, ph = normalise(x, px, beta, alpha)
r = np.hypot(xh, ph)
assert np.allclose((xh, ph), O.normalise(x, px, beta, alpha)), "compare with the formula above"
assert np.allclose(r, r[0], rtol=1e-10), "in normalised coordinates the radius must be constant"
assert np.allclose(r**2, D.cs_invariant(x, px, beta, alpha))
print("GOOD! Normalised coordinates: the ellipse is a circle with radius^2 = eps_i")

fig, axs = plt.subplots(1, 2, figsize=(11, 5))
axs[0].scatter(x * MM, px * MM, s=3, c=np.arange(len(x)), cmap="viridis")
axs[0].set_xlabel(r"$x$ [mm]"); axs[0].set_ylabel(r"$p_x$ [mrad]"); axs[0].set_title("physical coordinates")
axs[1].scatter(xh * MM, ph * MM, s=3, c=np.arange(len(x)), cmap="viridis")
for k in range(4):
    axs[1].annotate(str(k), (xh[k] * MM, ph[k] * MM), xytext=(5, 5), textcoords="offset points")
P.phase_space_labels(axs[1], "x", normalised=True); axs[1].set_aspect("equal")
axs[1].set_title("normalised: a circle, turns go clockwise")
plt.show()

## 2. Tune from the rotation angle

Define the phase $\phi$ by $\hat{x} = \sqrt{\varepsilon_i} \cos\phi$, $\hat{p}_x = -\sqrt{\varepsilon_i} \sin\phi$, i.e. $\phi = \arctan(−\hat{p}_x, \hat{x})$.
Each turn $\phi$ grows by $\mu = 2\pi Q$ (modulo $2\pi$).

**Task.** Implement `tune_from_angle`: compute $\phi$ for every turn, the turn-to-turn differences
wrapped into $[0, 2\pi)$ (use `% (2 * np.pi)`), and return their mean / $2\pi$.

In [ ]:
def tune_from_angle(xh: np.ndarray, ph: np.ndarray) -> float:
    """Fractional tune from the average rotation angle per turn."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
Q_angle = tune_from_angle(xh, ph)
print(f"Q from angles: {Q_angle:.12f}  (matrix: {Q_true:.12f})")
assert abs(Q_angle - Q_true) < 1e-6, "should agree to better than 1e-6"
print("GOOD! Tune recovered from the rotation angle")

## 3. Tune from a spectrum

In a real machine we only measure the **position** $x$ at a beam position monitor (BPM), once per
turn: $x_n = A \cos(2\pi Q n + \phi_0)$. The Fourier transform of these $N$ numbers has a peak at the
frequency $Q$ (in units of "oscillations per turn").

**Task.** Implement `tune_from_fft(x)`: subtract the mean, take `np.fft.rfft`, find the bin $k$ with
the largest magnitude and return $k$/$N$. The resolution is 1/$N$.

In [ ]:
def tune_from_fft(x: np.ndarray) -> float:
    """Tune from the position of the largest FFT peak (no interpolation)."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
Q_fft = tune_from_fft(x)
print(f"Q from FFT peak: {Q_fft:.6f}   error {abs(Q_fft - Q_true):.1e}  (resolution 1/N = {1 / len(x):.1e})")
assert abs(Q_fft - Q_true) <= 1 / len(x)
Q_naff = D.tune_naff(D.complex_signal(x, px, beta, alpha))
print(f"Q from refined spectrum (beamlib.diagnostics.tune_naff): {Q_naff:.12f}   error {abs(Q_naff - Q_true):.1e}")
assert abs(Q_naff - Q_true) < 1e-8
print("GOOD! FFT gives the tune to 1/N; refining the peak position gives it much more precisely")

`tune_naff` uses a window (to reduce leakage) and then searches for the exact frequency where the
spectrum peaks, instead of only looking at the FFT bins. This is the idea of NAFF, the algorithm used
for real tune measurements.

## 4. Why only the fractional tune? (aliasing)

The total tune of this ring is $Q_x \propto 2.137$: the particle oscillates a bit more than twice per turn.
But the BPM only sees it once per turn.

**Predict first.** Through the once-per-turn points, can you draw a slow cosine with 0.137 oscillations
per turn? And one with 1 − 0.137 = 0.863? And one with 2.137?

*Your prediction:*

In [ ]:
s, elem, Ms = O.cumulative_matrices(ringB, n_slices=10)
C = s[-1]
n_show = 6
z = res.coords[:n_show, :, 0]                       # state at B on turns 0..5
x_along = np.concatenate([(Ms[:-1] @ np.r_[zz, 0.0])[:, 0] for zz in z])
s_along = np.concatenate([s[:-1] / C + k for k in range(n_show)])

t = np.linspace(0, n_show - 1, 600)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(s_along, x_along * MM, color=COLORS["x"], lw=1, label=r"$x(s)$ along the ring ($Q$ = 2.137)")
ax.plot(np.arange(n_show), x[:n_show] * MM, "ko", ms=7, label="BPM at B, once per turn")
for q, ls in [(Q_true, "--"), (1 - Q_true, ":")]:
    # fit a cosine with frequency q through the samples
    a = np.linalg.lstsq(np.column_stack([np.cos(2 * np.pi * q * np.arange(n_show)),
                                         np.sin(2 * np.pi * q * np.arange(n_show))]), x[:n_show], rcond=None)[0]
    ax.plot(t, (a[0] * np.cos(2 * np.pi * q * t) + a[1] * np.sin(2 * np.pi * q * t)) * MM, ls,
            color=COLORS["accent"], label=rf"cosine with $Q$ = {q:.3f}")
ax.set_xlabel("time [turns]"); ax.set_ylabel("x [mm]"); ax.legend(loc="lower left", fontsize=8)
plt.show()

All three curves pass through every black point. Sampling once per turn cannot distinguish
$Q$ = 2.137 from 0.137 (they differ by a whole number of oscillations between samples) nor from
0.863 = 1 − 0.137 (a cosine running "backwards"). This is **aliasing**, the same effect that makes
wheels in films seem to turn backwards.

* The **integer part** can only be found by looking at many places around the ring (many BPMs) and
  counting the oscillations, as we did in Exercise 4 with the phase advance.
* $Q$ vs 1 − $Q$ can be resolved if we also know $p_x$ (or use two BPMs): the complex signal
  $\hat{x} − i \hat{p}$ rotates in one direction only, and its spectrum has a single peak.

## 5. Explore: a slider for $Q$

The plot below shows a linear ring with the same $\beta$ and $\alpha$ as location B, and the tune you choose.
Left: the first 12 turns in physical and normalised coordinates. Right: the spectrum of the real
signal $x$ and of the complex signal $\hat{x} − i \hat{p}$.

Try $Q$ = 0.25 (the particle comes back after 4 turns), $Q$ = 0.5, $Q$ close to 0, and $Q$ = 0.137
vs 0.863.

In [ ]:
def show_tune(Q):
    ring_q = L.HenonRing(Qx=Q, Qy=0.3, betx=beta, alfx=alpha)
    r = T.track(np.array([[1.5e-3], [0.2e-3], [0], [0]]), ring_q.one_turn_map, 255)
    xx, pp = r.x[:, 0], r.px[:, 0]
    xhh, phh = O.normalise(xx, pp, beta, alpha)
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
    n = 12
    axs[0].plot(xx[:n] * MM, pp[:n] * MM, "o-", color=COLORS["x"], ms=4, lw=0.5)
    axs[0].set_xlabel("x [mm]"); axs[0].set_ylabel("px [mrad]"); axs[0].set_title("physical")
    axs[1].plot(xhh[:n] * MM, phh[:n] * MM, "o-", color=COLORS["x"], ms=4, lw=0.5)
    for k in range(n):
        axs[1].annotate(str(k), (xhh[k] * MM, phh[k] * MM), xytext=(4, 4), textcoords="offset points", fontsize=8)
    P.phase_space_labels(axs[1], "x", True); axs[1].set_aspect("equal"); axs[1].set_title("normalised")
    f = np.fft.fftfreq(len(xx))
    order = np.argsort(f)
    w = np.hanning(len(xx))
    axs[2].plot(f[order], np.abs(np.fft.fft(xx * w))[order] / np.abs(np.fft.fft(xx * w)).max(),
                color=COLORS["x"], label="spectrum of x (real)")
    zc = xhh - 1j * phh
    axs[2].plot(f[order] % 1, np.abs(np.fft.fft(zc * w))[order] / np.abs(np.fft.fft(zc * w)).max(),
                color=COLORS["accent"], linestyle='--', label=r"spectrum of $\hat x - i\hat p$ (complex)")
    axs[2].axvline(Q, color="k", ls=":", lw=1)
    axs[2].set_xlim(-0.5, 1); axs[2].set_xlabel("frequency [1/turn]"); axs[2].legend(fontsize=8)
    axs[2].set_title(f"Q = {Q:.3f}: tune_naff gives {D.tune_naff(zc):.4f}")
    plt.show()

interactive_plot(show_tune, Q=(0.0, 1.0, 0.005, 0.137))

## Reflection questions

1. In normalised coordinates all particles rotate by the same angle every turn. What would change
   if the rotation angle depended on the radius? (Keep this question in mind for Block 4.)
2. Why does the FFT peak of the real signal $x$ for $Q$ = 0.863 appear at 0.137?
3. What happens to the turn-by-turn points when $Q$ = 0.25 exactly? Why is that dangerous if the
   ring has a magnet error?